# 🚀 LoRA Fine-Tuning Open-Source LLM (Llama 3.2) on Amazon Customer Support
### Dual Nvidia Tesla T4 GPU Cloud Pipeline: QLoRA Fine-Tuning, 7-Intent Classification & Escalation Engine

**Candidate**: Rohan Alex Bimal | Hiver SDE Intern Challenge (12 LPA | 2027 Batch)  
**Architecture**: Llama-3.2-1B-Instruct + 4-bit NF4 Quantization + LoRA (Rank 16, Alpha 32)  
**Dataset**: `@AmazonHelp` Conversational ChatML Dataset (`rohanalexbimal/amazon-support-chatml-5k`)


In [ ]:
# [1] Environment Verification & Dual T4 GPU Check
import os, sys, time, json, math, re, csv, glob
import torch

print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Count       : {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'GPU [{i}]        : {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / (1024**3):.2f} GB VRAM, Compute {torch.cuda.get_device_capability(i)})')
else:
    print('Warning: Running on CPU.')


In [ ]:
# [2] Install Fine-Tuning & Evaluation Stack
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets rouge-score tabulate
print('Fine-tuning dependencies installed successfully.')


In [ ]:
# [3] Ingest Conversational JSONL Dataset
print('Scanning /kaggle/input for dataset files...')
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.jsonl') or f.endswith('.csv'):
            print(f'Discovered: {os.path.join(root, f)} ({os.path.getsize(os.path.join(root, f)) / (1024*1024):.2f} MB)')

jsonl_candidates = glob.glob('/kaggle/input/**/*.jsonl', recursive=True)
csv_candidates = glob.glob('/kaggle/input/**/twcs.csv', recursive=True) + glob.glob('/kaggle/input/**/*.csv', recursive=True)

train_conversations = []

if jsonl_candidates:
    target_jsonl = jsonl_candidates[0]
    print(f'\nLoading ChatML JSONL dataset: {target_jsonl}')
    with open(target_jsonl, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if not line.strip(): continue
            try:
                record = json.loads(line)
                if 'messages' in record and len(record['messages']) >= 2:
                    train_conversations.append(record['messages'])
                elif 'query' in record and 'resolution' in record:
                    train_conversations.append([
                        {'role': 'system', 'content': 'You are AmazonHelp, a customer support agent on Twitter. Be empathetic, concise (<=280 chars), and provide actionable resolution links.'},
                        {'role': 'user', 'content': record['query']},
                        {'role': 'assistant', 'content': record['resolution']}
                    ])
            except Exception:
                pass
elif csv_candidates:
    target_csv = csv_candidates[0]
    print(f'\nReconstructing conversations from CSV: {target_csv}')
    amazon_replies = {}
    with open(target_csv, 'r', encoding='utf-8', errors='ignore') as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if len(row) >= 7 and row[1].strip().lower() == 'amazonhelp' and row[6]:
                amazon_replies[row[6]] = row[4]
                if len(amazon_replies) >= 15000: break
    with open(target_csv, 'r', encoding='utf-8', errors='ignore') as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if len(row) >= 7 and row[0] in amazon_replies:
                c_text = row[4]
                if sum(1 for c in c_text if c.isascii()) / max(len(c_text), 1) > 0.85 and len(c_text) >= 20:
                    train_conversations.append([
                        {'role': 'system', 'content': 'You are AmazonHelp, a customer support agent on Twitter. Be empathetic, concise (<=280 chars), and provide actionable resolution links.'},
                        {'role': 'user', 'content': c_text},
                        {'role': 'assistant', 'content': amazon_replies[row[0]]}
                    ])
                if len(train_conversations) >= 5000: break

if not train_conversations:
    print('Using embedded seed conversations for standalone execution.')
    train_conversations = [
        [
            {'role': 'system', 'content': 'You are AmazonHelp, a customer support agent on Twitter. Be empathetic, concise (<=280 chars), and provide actionable resolution links.'},
            {'role': 'user', 'content': '@AmazonHelp Where is my package tracking TBA982348123019? Delayed 2 days!'},
            {'role': 'assistant', 'content': '@user We are sorry for the delay! Please check live tracking at amazon.com/your-orders. If still not updated after 48h, please DM us your order ID.'}
        ],
        [
            {'role': 'system', 'content': 'You are AmazonHelp, a customer support agent on Twitter. Be empathetic, concise (<=280 chars), and provide actionable resolution links.'},
            {'role': 'user', 'content': '@AmazonHelp Returned my item at Whole Foods, when do I get my refund?'},
            {'role': 'assistant', 'content': '@user Refunds typically reflect in 3-5 business days from carrier drop-off. You can track refund progress at amazon.com/returns.'}
        ],
        [
            {'role': 'system', 'content': 'You are AmazonHelp, a customer support agent on Twitter. Be empathetic, concise (<=280 chars), and provide actionable resolution links.'},
            {'role': 'user', 'content': '@AmazonHelp My card was charged $139 for Prime renewal but I cancelled!'},
            {'role': 'assistant', 'content': '@user We would like to look into this billing charge for you. Please send us a private DM with your account email so our billing team can assist.'}
        ]
    ]

print(f'\nSuccessfully prepared {len(train_conversations):,} conversational training examples!')
print(f'Sample User Input : {train_conversations[0][1]["content"]}')
print(f'Sample Agent Reply: {train_conversations[0][2]["content"]}')


In [ ]:
# [4] Load Base Model & Tokenizer (4-bit QLoRA)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = 'unsloth/Llama-3.2-1B-Instruct'
print(f'Loading Base Model: {MODEL_ID} in 4-bit Quantization (QLoRA)...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    print(f'Successfully loaded {MODEL_ID} into GPU memory.')
except Exception as e:
    print(f'Primary model load note ({e}), trying Qwen/Qwen2.5-0.5B-Instruct fallback...')
    MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    print(f'Successfully loaded fallback model {MODEL_ID}.')


In [ ]:
# [5] Baseline Zero-Shot Inference (Pre-Fine-Tuning)
test_prompts = [
    'Where is my package tracking TBA982348123019? Was supposed to arrive yesterday!',
    'My credit card was charged $139 for Prime renewal but I cancelled 2 weeks ago!',
    'My Fire TV stick remote is frozen and wont pair with my TV.'
]

def generate_reply(m, tok, query):
    messages = [
        {'role': 'system', 'content': 'You are AmazonHelp, a verified customer support agent on Twitter. Respond politely, accurately, and concisely within 280 characters.'},
        {'role': 'user', 'content': query}
    ]
    formatted_prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(formatted_prompt, return_tensors='pt').to(m.device)
    with torch.no_grad():
        outputs = m.generate(
            **inputs,
            max_new_tokens=90,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tok.pad_token_id
        )
    full_output = tok.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return full_output.strip()

print('='*80)
print('             PRE-FINE-TUNING ZERO-SHOT BASELINE REPLIES')
print('='*80)
baseline_replies = {}
for q in test_prompts:
    res = generate_reply(model, tokenizer, q)
    baseline_replies[q] = res
    print(f'Customer Tweet : {q}')
    print(f'Base Response  : {res}')
    print(f'Char Length    : {len(res)} (Limit <= 280)')
    print('-'*80)


In [ ]:
# [6] Setup LoRA Configuration & PEFT Model
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, peft_config)
print('LoRA Adapter Configuration:')
model.print_trainable_parameters()


In [ ]:
# [7] Format Training Data & Execute LoRA Fine-Tuning
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

formatted_texts = []
sample_conversations = train_conversations[:500]
for msgs in sample_conversations:
    txt = tokenizer.apply_chat_template(msgs, tokenize=False)
    formatted_texts.append({'text': txt})

train_dataset = Dataset.from_list(formatted_texts)

def tokenize_fn(examples):
    result = tokenizer(examples['text'], truncation=True, max_length=256, padding='max_length')
    result['labels'] = result['input_ids'].copy()
    return result

tokenized_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
print(f'Tokenized dataset ready with {len(tokenized_dataset)} samples.')

output_dir = './amazon_support_lora_output'
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    logging_steps=10,
    max_steps=60,
    fp16=True,
    save_strategy='no',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors='pt', padding=True)
)

print('\nStarting LoRA Fine-Tuning on Nvidia Tesla T4 GPU...')
start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time
print(f'Training completed successfully in {elapsed:.1f}s ({elapsed/60:.2f} min)!')
print(f'Final Training Loss: {train_result.training_loss:.4f}')

# Save LoRA Adapter
adapter_save_path = './amazon_help_lora_adapter'
model.save_pretrained(adapter_save_path)
tokenizer.save_pretrained(adapter_save_path)
print(f'LoRA adapter weights saved to: {adapter_save_path}')


In [ ]:
# [8] Post Fine-Tuning Inference & Comparative Audit
from tabulate import tabulate

model.eval()
print('='*85)
print('         POST-FINE-TUNING COMPARATIVE AUDIT (BASE vs LORA ADAPTER)')
print('='*85)

comparison_rows = []
for q in test_prompts:
    lora_reply = generate_reply(model, tokenizer, q)
    base_reply = baseline_replies[q]
    
    print(f'\n[Customer Tweet] : {q}')
    print(f'[Base Model]     : {base_reply} ({len(base_reply)} chars)')
    print(f'[LoRA Fine-Tuned]: {lora_reply} ({len(lora_reply)} chars)')
    print('-' * 85)
    
    comparison_rows.append([
        q[:35] + '...',
        base_reply[:40] + '...',
        lora_reply[:40] + '...',
        f'{len(base_reply)} chars',
        f'{len(lora_reply)} chars',
        'PASS (<=280)' if len(lora_reply) <= 280 else 'FAIL'
    ])

print('\n' + tabulate(comparison_rows, headers=['Customer Query', 'Base Reply', 'LoRA Reply', 'Base Len', 'LoRA Len', '280 Limit'], tablefmt='grid'))


In [ ]:
# [9] Full System Integration: 7-Intent Classifier + RAG + Escalation + LoRA Agent
INTENT_TAXONOMY = {
    'ORDER_STATUS_DELIVERY': {
        'keywords': ['tracking', 'track', 'delivery', 'delivered', 'package', 'where is', 'delayed', 'late', 'carrier', 'tba', 'shipping', 'transit', 'porch', 'driver'],
        'policy': 'Direct to amazon.com/your-orders, explain 24h window, DM if >48h missing.'
    },
    'REFUND_AND_RETURNS': {
        'keywords': ['refund', 'return', 'returned', 'kohl', 'whole foods', 'ups drop', 'money back', 'reimburse', 'label', 'qr code'],
        'policy': 'Direct to amazon.com/returns, mention 3-5 business day refund window.'
    },
    'DAMAGED_DEFECTIVE_ITEM': {
        'keywords': ['damaged', 'broken', 'defective', 'cracked', 'shattered', 'leaking', 'expired', 'scratched', 'wrong item', 'faulty'],
        'policy': 'Offer instant replacement via amazon.com/returns without shipping broken glass back.'
    },
    'ACCOUNT_ACCESS_SECURITY': {
        'keywords': ['hacked', 'unauthorized', 'locked out', '2fa', 'otp', 'password', 'phishing', 'scam', 'suspend', 'fraud', 'compromised'],
        'policy': 'Never ask credentials in public. Escalate immediately to Account Security Team.'
    },
    'BILLING_AND_PRIME': {
        'keywords': ['prime', 'charged', 'billing', 'subscription', 'renewal', 'duplicate charge', 'invoice', 'payment failed', 'membership'],
        'policy': 'Direct to amazon.com/gp/primecentral for cancellations or DM for billing audit.'
    },
    'PRODUCT_TROUBLESHOOTING': {
        'keywords': ['kindle', 'fire tv', 'alexa', 'echo', 'remote', 'wifi', 'bluetooth', 'boot loop', 'frozen', 'stream', 'troubleshoot', 'reset'],
        'policy': 'Provide 40-second power cycle instructions and link to amazon.com/devicesupport.'
    },
    'FEEDBACK_AND_GENERAL': {
        'keywords': ['compliment', 'feedback', 'shoutout', 'praise', 'app update', 'smile', 'thank you', 'thanks'],
        'policy': 'Thank user, forward feedback to station/development team.'
    }
}

class FullCustomerSupportAgent:
    def __init__(self, m, tok):
        self.model = m
        self.tokenizer = tok
        
    def classify_intent(self, text):
        t_lower = text.lower()
        scores = {k: 0 for k in INTENT_TAXONOMY}
        for intent, data in INTENT_TAXONOMY.items():
            for kw in data['keywords']:
                if kw in t_lower: scores[intent] += 1
        top_intent = max(scores, key=scores.get)
        conf = 0.90 if scores[top_intent] > 0 else 0.50
        if scores[top_intent] == 0:
            top_intent = 'ORDER_STATUS_DELIVERY' if 'order' in t_lower else 'FEEDBACK_AND_GENERAL'
        return top_intent, conf
        
    def evaluate_escalation(self, text, intent, conf):
        t = text.lower()
        if intent == 'ACCOUNT_ACCESS_SECURITY':
            return True, 'Mandatory security & account lockout escalation.'
        if any(w in t for w in ['police', 'lawyer', 'legal', 'smoke', 'fire', 'exploded', 'injured']):
            return True, 'Critical physical safety hazard or legal dispute requiring Executive Team.'
        if any(w in t for w in ['3 times', 'already called', 'still waiting', 'second time', '2 weeks']):
            return True, 'Repeated unresolved friction requiring supervisor review.'
        if any(w in t for w in ['look into my account', 'check my order', 'unauthorized charge', 'stolen']):
            return True, 'Requires private account-level order lookup via secure DM.'
        if conf < 0.70:
            return True, 'Low classification confidence requiring human triage.'
        return False, 'Standard query eligible for automated self-service under brand SOP.'
        
    def run(self, tweet):
        intent, conf = self.classify_intent(tweet)
        should_esc, reason = self.evaluate_escalation(tweet, intent, conf)
        policy = INTENT_TAXONOMY[intent]['policy']
        prompt_with_rag = f'Customer Query: {tweet}\nPolicy Rule: {policy}\nDraft grounded @AmazonHelp tweet reply <= 280 chars:'
        reply = generate_reply(self.model, self.tokenizer, prompt_with_rag)
        return {
            'tweet': tweet,
            'intent': intent,
            'confidence': conf,
            'escalate': should_esc,
            'reason': reason,
            'reply': reply,
            'length': len(reply)
        }

agent = FullCustomerSupportAgent(model, tokenizer)

test_suite = [
    'Where is my package tracking TBA982348123019? Was supposed to arrive yesterday!',
    'My credit card was charged $139 for Prime renewal but I cancelled 2 weeks ago!',
    'Account was hacked and someone changed my email address and password!',
    'How do I return an unopened coffee maker at Whole Foods?'
]

print('='*85)
print('             END-TO-END PRODUCTION PIPELINE AUDIT (WITH LORA)')
print('='*85)
for q in test_suite:
    res = agent.run(q)
    print(f'Customer Tweet   : {res["tweet"]}')
    print(f'Predicted Intent : {res["intent"]} (Conf: {res["confidence"]})')
    print(f'Escalation State : {"ESCALATED TO HUMAN" if res["escalate"] else "AUTO-RESOLVED (SELF-SERVICE)"}')
    print(f'Stated Reason    : {res["reason"]}')
    print(f'LoRA Reply       : {res["reply"]}')
    print(f'Character Count  : {res["length"]} / 280 limit')
    print('-' * 85)

print('\nPipeline execution, LoRA fine-tuning, and audit completed with 0 errors!')
